# DB3 v5 — parallel Gemma 31B DB-COMP classifier

This notebook replaces the earlier **exclusion-only** classifier with a document-level classifier built around the actual research question:

> **Is the supplied document itself a substantive operative European Commission antitrust act that belongs in the research corpus, or is it surrounding / non-substantive material that should be excluded?**

## Core design

The model separately returns:

1. `inclusion_status`: `include`, `exclude`, or `unclear`
2. `document_class`: what kind of document this actually is
3. `decision_type`: the substantive Commission-act subtype when applicable
4. confidence, evidence, reason, and manual-review flag

The Python layer then applies an **asymmetric safety rule**:

- high-confidence substantive act → `KEEP`
- high-confidence exclusion → `EXCLUDE`
- lower-confidence exclusion → `REVIEW`
- unclear / failed / missing text → `REVIEW`

This is intentionally conservative: it is preferable to manually review an extra press release than to silently discard a genuine Commission decision.

## Parallel execution

The notebook uses the same architecture that worked well in the InfoCuria classifier:

- `asyncio`
- `httpx.AsyncClient`
- configurable concurrent requests (default: 6)
- retries with exponential backoff
- append-as-you-go JSONL checkpointing
- durable `flush` + `fsync`
- `update`, `redo_all`, and `rebuild_csv` modes

The default model is:

`google/gemma-4-31B-it`


## FAST changes

This version keeps the same taxonomy and conservative decision rules, but is optimized for throughput:

- **280 output tokens** instead of 900
- only **4 structured model fields** instead of long evidence/reason strings
- **12,000 input characters** instead of 22,000
- minimal metadata in the prompt
- **1 retry** instead of 2
- token-limit responses are marked `REVIEW` instead of repeating the same long generation
- default `RUN_MODE="update"` so an existing v2 checkpoint can be resumed


## BALANCED profile

This version sits between the original verbose classifier and the FAST classifier.

It intentionally spends more compute only where it adds audit value:

- **120,000 document characters** (50k head + 50k tail for longer documents)
- **600 output-token cap**
- keeps `inclusion_status`, `document_class`, `decision_type`, and `confidence`
- restores a **short evidence snippet**
- restores a **one-sentence reason**
- retains **6 concurrent requests**
- retains live `LENGTH / RETRY / FAILED / NO INPUT` diagnostics
- retains the conservative KEEP / EXCLUDE / REVIEW thresholds

The goal is not to force a fixed runtime. The goal is a more auditable classification while remaining substantially lighter than the InfoCuria Court-reference task.


## v5 document-context rule

- If the extracted document is **120,000 characters or shorter**, Gemma receives the **entire document**.
- If the document is **longer than 120,000 characters**, Gemma receives the **first 50,000 characters + last 50,000 characters**, with the middle omitted.

All other v4 classifier logic, parallelism, audit fields, thresholds, retries, and live diagnostics are unchanged.


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Iterable, Optional, List
from datetime import datetime, timezone

import asyncio
import json
import os
import re

import pandas as pd
import httpx

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable: Iterable, **kwargs):
        return iterable

print("pandas:", pd.__version__)
print("httpx:", httpx.__version__)


pandas: 2.3.2
httpx: 0.28.1


## 1. Configuration

### Run modes

- `redo_all`: delete the v2 JSONL checkpoint and classify every row again
- `update`: keep successful checkpointed rows and retry missing / failed / unclear rows
- `rebuild_csv`: make no LLM calls; rebuild the CSV outputs from the existing checkpoint

### Versioned outputs

v2 writes new files and does **not** overwrite the earlier DB3 outputs.


In [2]:
# -------------------------------------------------------------------
# Project paths
# -------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "notebooks" and candidate.parent.name == "code":
            return candidate.parent.parent
        if (candidate / "output").exists() and (candidate / "data").exists():
            return candidate
        if candidate.name == "eccjeu":
            return candidate
    return start

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
COM_DB_COMP_OUTPUT_DIR = PROJECT_ROOT / "output" / "com_db_comp"

DB_COMP_MANIFEST_PATH = COM_DB_COMP_OUTPUT_DIR / "db_comp_manifest.csv"
CLEAN_FILE_MANIFEST_PATH = COM_DB_COMP_OUTPUT_DIR / "db_comp_clean_file_manifest.csv"

JSONL_RESULTS_PATH = COM_DB_COMP_OUTPUT_DIR / "db_comp_document_classification_v2.jsonl"
METADATA_MANIFEST_PATH = COM_DB_COMP_OUTPUT_DIR / "db_comp_metadata_manifest_v2.csv"
SUMMARY_PATH = COM_DB_COMP_OUTPUT_DIR / "db_comp_document_classification_summary_v2.csv"
RUN_INFO_PATH = COM_DB_COMP_OUTPUT_DIR / "db_comp_document_classification_run_info_v2.json"

COM_DB_COMP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_COMP_MANIFEST_PATH:", DB_COMP_MANIFEST_PATH)
print("CLEAN_FILE_MANIFEST_PATH:", CLEAN_FILE_MANIFEST_PATH)
print("JSONL_RESULTS_PATH:", JSONL_RESULTS_PATH)
print("METADATA_MANIFEST_PATH:", METADATA_MANIFEST_PATH)


PROJECT_ROOT: /home/edik/projects/eccjeu
DB_COMP_MANIFEST_PATH: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_manifest.csv
CLEAN_FILE_MANIFEST_PATH: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_clean_file_manifest.csv
JSONL_RESULTS_PATH: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_document_classification_v2.jsonl
METADATA_MANIFEST_PATH: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_metadata_manifest_v2.csv


In [3]:
# -------------------------------------------------------------------
# Local Gemma / vLLM endpoint
# -------------------------------------------------------------------
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")
MODEL = os.getenv("VLLM_MODEL", "google/gemma-4-31B-it")

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 600
REQUEST_TIMEOUT_SECONDS = int(os.getenv("REQUEST_TIMEOUT_SECONDS", "3600"))

# Parallelism. Six concurrent requests worked well in the InfoCuria pipeline.
CONCURRENT_REQUESTS = int(os.getenv("CONCURRENT_REQUESTS", "6"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "1"))


# -------------------------------------------------------------------
# Live diagnostics
# -------------------------------------------------------------------
# Print only exceptional events, not every successful classification.
PRINT_LIVE_DIAGNOSTICS = True

# Optional: also print rows that technically succeeded but end up in REVIEW
# because of uncertainty / confidence. Leave False for a clean notebook.
PRINT_REVIEW_ROWS = False

# -------------------------------------------------------------------
# Run control
# -------------------------------------------------------------------
RUN_MODE = os.getenv("RUN_MODE", "update").strip().lower()
VALID_RUN_MODES = {"redo_all", "update", "rebuild_csv"}

TEST_MODE = False
TEST_LIMIT = 10

# Text cap: enough to identify document type while keeping inference manageable.
MAX_TEXT_CHARS = 120000
HEAD_CHARS = 50000
TAIL_CHARS = 50000

# -------------------------------------------------------------------
# Conservative operational thresholds
# -------------------------------------------------------------------
# Exclusion is intentionally harder than inclusion because a false exclusion
# can silently remove a genuine Commission decision from the research corpus.
AUTO_EXCLUDE_CONFIDENCE = 0.90
AUTO_KEEP_CONFIDENCE = 0.70

# In update mode, successful unclear/review rows can optionally be retried.
RECLASSIFY_UNCLEAR = True
RECLASSIFY_REVIEW = False
RECLASSIFY_LOW_CONFIDENCE = False
RECLASSIFY_CONFIDENCE_BELOW = 0.60

if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}")

print("Model:", MODEL)
print("RUN_MODE:", RUN_MODE)
print("Concurrent requests:", CONCURRENT_REQUESTS)
print("Auto-exclude threshold:", AUTO_EXCLUDE_CONFIDENCE)
print("Auto-keep threshold:", AUTO_KEEP_CONFIDENCE)


Model: google/gemma-4-31B-it
RUN_MODE: update
Concurrent requests: 6
Auto-exclude threshold: 0.9
Auto-keep threshold: 0.7


## 2. Load and merge manifests

The DB-COMP manifest remains the metadata spine. The clean-file manifest supplies the extracted text / markdown paths.

The merge key remains:

- `dbcomp_document_id`
- `case_number`


In [4]:
def read_csv_required(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Could not find {name}: {path}")
    df = pd.read_csv(path, low_memory=False)
    print(f"{name}: {len(df):,} rows | {len(df.columns)} columns")
    return df

def clean_scalar(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()

def normalize_id_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc in {"document_id", "db_comp_document_id", "dbcomp_id"} and "dbcomp_document_id" not in df.columns:
            rename[c] = "dbcomp_document_id"
        if lc in {"modified_case_number", "case_no", "case_reference"} and "case_number" not in df.columns:
            rename[c] = "case_number"
    if rename:
        df = df.rename(columns=rename)
    for col in ["dbcomp_document_id", "case_number"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].map(clean_scalar)
    return df

db_comp_manifest = normalize_id_cols(
    read_csv_required(DB_COMP_MANIFEST_PATH, "db_comp_manifest")
)
clean_file_manifest = normalize_id_cols(
    read_csv_required(CLEAN_FILE_MANIFEST_PATH, "db_comp_clean_file_manifest")
)

print("\ndb_comp_manifest columns:")
print(list(db_comp_manifest.columns))
print("\nclean_file_manifest columns:")
print(list(clean_file_manifest.columns))


db_comp_manifest: 833 rows | 19 columns
db_comp_clean_file_manifest: 833 rows | 49 columns

db_comp_manifest columns:
['dbcomp_document_id', 'case_number', 'source', 'page', 'page_url', 'result_index', 'db_comp_case_number_saved', 'db_comp_case_number_extracted_from_text', 'document_name', 'decision_date', 'pdf_filename', 'dbcomp_document_url', 'file_url', 'file_name', 'file_extension', 'raw_links_json', 'raw_result_text', 'scrape_status', 'scrape_error']

clean_file_manifest columns:
['document_id', 'dbcomp_document_id', 'case_number', 'title_for_processing', 'source_url_for_processing', 'raw_file_path', 'raw_file_type', 'file_format', 'processing_eligible', 'processing_skip_reason', 'cleaning_target', 'clean_success', 'extraction_version', 'quality_flag', 'quality_score', 'quality_reasons', 'needs_manual_review', 'n_chars_raw', 'n_chars_clean', 'n_pages', 'selected_method', 'extraction_engine', 'mineru_used', 'candidate_count', 'candidate_methods', 'candidate_scores', 'second_best_sc

In [5]:
# Select only clean-file columns needed downstream.
path_cols = [
    "dbcomp_document_id", "case_number",
    "raw_file_path", "raw_path", "local_path", "final_local_path",
    "clean_text_path", "text_path", "final_text_path",
    "markdown_path", "final_markdown_path",
    "extraction_status", "conversion_status", "extraction_engine",
    "n_chars", "n_words", "quality_flag",
    "candidate_for_llm", "candidate_for_classification",
]
clean_keep = [c for c in path_cols if c in clean_file_manifest.columns]
clean_slim = clean_file_manifest[clean_keep].copy()

clean_slim = clean_slim.drop_duplicates(
    subset=["dbcomp_document_id", "case_number"],
    keep="first",
)

work = db_comp_manifest.merge(
    clean_slim,
    on=["dbcomp_document_id", "case_number"],
    how="left",
    suffixes=("", "_clean"),
)

print("Merged work rows:", len(work))
if "clean_text_path" in work.columns:
    print("Rows with clean_text_path:", work["clean_text_path"].notna().sum())
display(work.head(5))


Merged work rows: 833
Rows with clean_text_path: 833


,dbcomp_document_id,case_number,source,page,page_url,result_index,db_comp_case_number_saved,db_comp_case_number_extracted_from_text,document_name,decision_date,...,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error,raw_file_path,clean_text_path,markdown_path,extraction_engine,quality_flag
0,1061,93,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,1,93,93,European Machine Tool Exhibitions (EEMO),13 Mar 1969,...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",93 European Machine Tool Exhibitions (EEMO) CE...,parsed,NaN,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,pdf_pymupdf_text_sorted,ok
1,1064,399904,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,2,399904,399904,Rechargeable battery,12 Dec 2016,...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",399904 Rechargeable battery 399904-rechargeabl...,parsed,NaN,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,pdf_pymupdf_blocks_sorted,ok
2,1065,40481,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,3,40481,40481,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40481 Occupant Safety Systems (II) supplied to...,parsed,NaN,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,pdf_mineru,ok
3,1066,40360,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,4,40360,40360,Production and distribution of audiobooks,19 Jan 2017,...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40360 Production and distribution of audiobook...,parsed,NaN,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,pdf_pdftotext_layout,ok
4,1067,40291,db-comp.eu,1,https://db-comp.eu/?slotName=Slot_Main_3&Searc...,5,40291,40291,Aquatrend,21 Jan 2016,...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40291 Aquatrend 40291-Aquatrend-(Jan-2016).pdf...,parsed,NaN,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,pdf_pymupdf_text_sorted,ok


## 3. Resolve the best text input

Preference order:

1. cleaned text
2. alternate final/text path
3. markdown fallback

For long documents, Gemma receives the beginning and end of the document. This is useful for document-type classification because formal headings tend to appear near the start and operative provisions/signatures near the end.


In [6]:
def resolve_path(value: Any) -> str:
    value = clean_scalar(value)
    if not value:
        return ""

    p = Path(value)
    candidates = []

    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([
            PROJECT_ROOT / p,
            NOTEBOOK_DIR / p,
            COM_DB_COMP_OUTPUT_DIR / p,
        ])

    for cand in candidates:
        if cand.exists() and cand.is_file():
            return str(cand.resolve())

    return ""

def first_existing_path(row: pd.Series, cols: List[str]) -> tuple[str, str]:
    for col in cols:
        if col in row.index:
            p = resolve_path(row.get(col))
            if p:
                return p, col
    return "", ""

text_candidates = [
    "clean_text_path",
    "final_text_path",
    "text_path",
    "markdown_path",
    "final_markdown_path",
]

resolved = work.apply(
    lambda r: first_existing_path(r, text_candidates),
    axis=1,
)

work["llm_input_path"] = [x[0] for x in resolved]
work["llm_input_path_source"] = [x[1] for x in resolved]

print("Input path sources:")
print(
    work["llm_input_path_source"]
    .replace("", "missing")
    .value_counts(dropna=False)
)


Input path sources:
llm_input_path_source
clean_text_path    833
Name: count, dtype: int64


## 4. Classification taxonomy and prompt

### Research rule

The classifier distinguishes **legal/document identity** from **research inclusion**.

For example, a rejection-of-complaint document may formally be a Commission decision, but under this corpus definition it is still classified as `exclude`.

### Most important prompt rule

> **Classify the supplied document itself. Do not classify it as a Commission decision merely because it describes, summarizes, announces, quotes, or refers to a Commission decision.**

### In-scope substantive acts

The corpus aims to retain substantive operative Commission antitrust acts under Articles **101, 102 and 106 TFEU and their historical predecessors**, including final and interim acts.


In [7]:
INCLUSION_STATUSES = [
    "include",
    "exclude",
    "unclear",
]

DOCUMENT_CLASSES = [
    "substantive_commission_act",
    "press_release_or_memo",
    "rejection_of_complaint",
    "market_test_notice",
    "informal_guidance_letter",
    "cover_or_transmission_letter",
    "summary_or_publication_notice",
    "hearing_officer_report",
    "advisory_committee_opinion",
    "closure_or_administrative_notice",
    "other_non_substantive_document",
    "outside_scope",
    "unclear",
]

DECISION_TYPES = [
    "infringement_or_prohibition",
    "commitment",
    "exemption_or_negative_clearance",
    "interim_measures",
    "readoption_or_amendment",
    "fine_or_periodic_penalty",
    "trustee_approval",
    "other_substantive_commission_act",
    "not_applicable",
    "unclear",
]

# FAST first-pass schema: only fields needed for KEEP / EXCLUDE / REVIEW.
RESPONSE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "inclusion_status": {
            "type": "string",
            "enum": INCLUSION_STATUSES,
        },
        "document_class": {
            "type": "string",
            "enum": DOCUMENT_CLASSES,
        },
        "decision_type": {
            "type": "string",
            "enum": DECISION_TYPES,
        },
        "confidence": {
            "type": "number",
            "minimum": 0.0,
            "maximum": 1.0,
        },
        "evidence_snippet": {
            "type": "string",
        },
        "reason": {
            "type": "string",
        },
    },
    "required": [
        "inclusion_status",
        "document_class",
        "decision_type",
        "confidence",
        "evidence_snippet",
        "reason",
    ],
    "additionalProperties": False,
}

SYSTEM_PROMPT = """
Classify the supplied European Commission DB-COMP document itself.

Corpus: substantive operative Commission antitrust acts under Articles 101, 102
and 106 TFEU and historical predecessors (including 85/86/90 EEC).

IMPORTANT: classify THIS DOCUMENT, not a decision it merely discusses,
announces, quotes or summarizes.

INCLUDE as substantive_commission_act:
- infringement/prohibition decisions
- commitment decisions
- exemption/negative-clearance decisions
- interim-measures decisions
- readoption/amendment decisions
- fines/periodic-penalty decisions
- trustee-approval decisions
- other substantive operative Commission antitrust acts

EXCLUDE:
- press release or MEMO
- rejection of complaint (excluded by research-design choice even if formally a decision)
- Article 27(4) or other market-test notice
- informal guidance letter
- cover/transmission letter
- OJ or other summary/publication notice
- Hearing Officer report
- Advisory Committee opinion
- closure/administrative notice
- other non-substantive surrounding material
- legal material clearly outside the 101/102/106 antitrust scope

Key distinctions:
- press release about a decision = exclude; actual decision = include
- OJ summary = exclude; actual operative decision = include
- Article 27(4) market test = exclude; final Article 9 commitment decision = include
- interim-measures decision = include

If uncertain whether this document itself is the operative act, return unclear.
For excluded documents use decision_type="not_applicable".

AUDIT FIELDS:
- evidence_snippet: give one short decisive quote or near-verbatim phrase from the supplied document, ideally under 180 characters.
- reason: give one concise sentence explaining why THIS DOCUMENT is included, excluded, or unclear, ideally under 220 characters.
- Do not write a long legal analysis.

Return only the structured JSON requested by the schema.
""".strip()

def read_text_for_llm(path: str) -> str:
    if not path:
        return ""

    p = Path(path)

    try:
        text = p.read_text(encoding="utf-8", errors="replace")
    except Exception:
        try:
            text = p.read_text(errors="replace")
        except Exception:
            return ""

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()

    if len(text) <= MAX_TEXT_CHARS:
        return text

    return (
        text[:HEAD_CHARS]
        + "\n\n[... middle omitted ...]\n\n"
        + text[-TAIL_CHARS:]
    )

def first_nonempty(row: pd.Series, cols: List[str]) -> str:
    for col in cols:
        if col in row.index:
            val = clean_scalar(row.get(col))
            if val:
                return val
    return ""

def build_user_prompt(row: pd.Series) -> str:
    meta = {
        "dbcomp_document_id": clean_scalar(row.get("dbcomp_document_id")),
        "case_number": clean_scalar(row.get("case_number")),
        "case_title": first_nonempty(
            row,
            ["case_title", "case_name", "title"],
        ),
        "document_title": first_nonempty(
            row,
            ["document_title", "file_title", "title", "name"],
        ),
    }

    text = read_text_for_llm(
        clean_scalar(row.get("llm_input_path"))
    )

    return (
        "Metadata:\n"
        + json.dumps(meta, ensure_ascii=False)
        + "\n\nDocument text:\n"
        + text
    )


## 5. Response parsing, validation and operational decision rule

The model's semantic classification and the pipeline's operational action are kept separate.

### Operational rule

- `include` with confidence ≥ `AUTO_KEEP_CONFIDENCE` → `keep`
- `exclude` with confidence ≥ `AUTO_EXCLUDE_CONFIDENCE` → `exclude`
- everything else → `review`

This means Gemma can say "exclude" at 0.78 confidence without the pipeline automatically dropping the document.


In [8]:
def extract_json_object(text: str) -> dict:
    text = (text or "").strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(text[start:end + 1])

    raise ValueError(
        f"Could not parse JSON from response: {text[:500]}"
    )

def as_bool(x: Any) -> bool:
    if isinstance(x, bool):
        return x
    if isinstance(x, str):
        return x.strip().lower() in {"true", "yes", "1", "y"}
    return bool(x)

def as_float(x: Any, default: float = 0.0) -> float:
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def normalize_token(x: Any) -> str:
    return (
        clean_scalar(x)
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )


def validate_model_response(obj: dict) -> dict:
    inclusion_status = normalize_token(
        obj.get("inclusion_status")
    )
    document_class = normalize_token(
        obj.get("document_class")
    )
    decision_type = normalize_token(
        obj.get("decision_type")
    )

    if inclusion_status not in set(INCLUSION_STATUSES):
        inclusion_status = "unclear"

    if document_class not in set(DOCUMENT_CLASSES):
        document_class = "unclear"

    if decision_type not in set(DECISION_TYPES):
        decision_type = "unclear"

    if inclusion_status == "include":
        if (
            document_class != "substantive_commission_act"
            or decision_type in {"not_applicable", "unclear"}
        ):
            inclusion_status = "unclear"
            document_class = "unclear"
            decision_type = "unclear"

    elif inclusion_status == "exclude":
        if document_class in {"substantive_commission_act", "unclear"}:
            inclusion_status = "unclear"
            document_class = "unclear"
            decision_type = "unclear"
        else:
            decision_type = "not_applicable"

    else:
        inclusion_status = "unclear"
        document_class = "unclear"
        decision_type = "unclear"

    confidence = max(
        0.0,
        min(
            1.0,
            as_float(obj.get("confidence"), 0.0),
        ),
    )

    if (
        inclusion_status == "include"
        and confidence >= AUTO_KEEP_CONFIDENCE
    ):
        pipeline_action = "keep"

    elif (
        inclusion_status == "exclude"
        and confidence >= AUTO_EXCLUDE_CONFIDENCE
    ):
        pipeline_action = "exclude"

    else:
        pipeline_action = "review"

    return {
        "llm_inclusion_status": inclusion_status,
        "llm_document_class": document_class,
        "llm_decision_type": decision_type,
        "llm_confidence": confidence,
        "llm_evidence_snippet": clean_scalar(
            obj.get("evidence_snippet")
        )[:300],
        "llm_reason": clean_scalar(
            obj.get("reason")
        )[:400],
        "llm_manual_review": pipeline_action == "review",
        "pipeline_action": pipeline_action,
        "pipeline_include": pipeline_action == "keep",
        "pipeline_exclude": pipeline_action == "exclude",
    }

def fallback_review(reason: str) -> dict:
    return {
        "llm_inclusion_status": "unclear",
        "llm_document_class": "unclear",
        "llm_decision_type": "unclear",
        "llm_confidence": 0.0,
        "llm_evidence_snippet": "",
        "llm_reason": reason,
        "llm_manual_review": True,
        "pipeline_action": "review",
        "pipeline_include": False,
        "pipeline_exclude": False,
    }


## 6. JSONL checkpoint and job selection

The checkpoint is the audit trail.

v2 hardens the checkpoint so that:

- malformed/truncated lines are skipped with a warning
- each completed record is appended immediately
- a missing final newline from an interrupted write is repaired before the next append
- every append is flushed and synced to disk


In [9]:
def make_row_key(row: pd.Series) -> str:
    return (
        clean_scalar(row.get("dbcomp_document_id"))
        + "||"
        + clean_scalar(row.get("case_number"))
    )

work["classification_key"] = work.apply(
    make_row_key,
    axis=1,
)

if TEST_MODE:
    work_run = work.head(TEST_LIMIT).copy()
else:
    work_run = work.copy()

def load_existing_results(path: Path) -> Dict[str, dict]:
    results: Dict[str, dict] = {}

    if not path.exists():
        return results

    skipped = 0

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rec = json.loads(line)
            except Exception as exc:
                skipped += 1
                print(
                    f"WARNING: skipping malformed JSONL line "
                    f"{line_no}: {type(exc).__name__}: {exc}"
                )
                continue

            key = clean_scalar(
                rec.get("classification_key")
            )

            if key:
                results[key] = rec

    if skipped:
        print("Malformed checkpoint lines skipped:", skipped)

    return results

def append_jsonl_durable(path: Path, rec: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = (
        json.dumps(
            rec,
            ensure_ascii=False,
            default=str,
        )
        + "\n"
    ).encode("utf-8")

    # If an interrupted previous append left a truncated final line,
    # isolate it before appending the next valid JSON object.
    if path.exists() and path.stat().st_size > 0:
        with path.open("rb") as handle:
            handle.seek(-1, os.SEEK_END)
            last_byte = handle.read(1)

        if last_byte != b"\n":
            with path.open("ab") as handle:
                handle.write(b"\n")
                handle.flush()
                os.fsync(handle.fileno())

    with path.open("ab") as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())

if RUN_MODE == "redo_all" and JSONL_RESULTS_PATH.exists():
    JSONL_RESULTS_PATH.unlink()
    print(
        "Removed previous v2 JSONL because RUN_MODE=redo_all:",
        JSONL_RESULTS_PATH,
    )

existing_results = load_existing_results(
    JSONL_RESULTS_PATH
)

print("Existing checkpoint rows:", len(existing_results))

def should_classify(row: pd.Series) -> bool:
    if RUN_MODE == "rebuild_csv":
        return False

    if RUN_MODE == "redo_all":
        return True

    key = clean_scalar(
        row.get("classification_key")
    )
    rec = existing_results.get(key)

    if rec is None:
        return True

    if clean_scalar(rec.get("llm_error")):
        return True

    if (
        RECLASSIFY_UNCLEAR
        and clean_scalar(
            rec.get("llm_inclusion_status")
        )
        == "unclear"
    ):
        return True

    if (
        RECLASSIFY_REVIEW
        and clean_scalar(
            rec.get("pipeline_action")
        )
        == "review"
    ):
        return True

    if (
        RECLASSIFY_LOW_CONFIDENCE
        and as_float(
            rec.get("llm_confidence"),
            0.0,
        )
        < RECLASSIFY_CONFIDENCE_BELOW
    ):
        return True

    return False

todo = work_run[
    work_run.apply(
        should_classify,
        axis=1,
    )
].copy()

print("Rows available:", len(work_run))
print("Rows to classify:", len(todo))


Existing checkpoint rows: 0
Rows available: 833
Rows to classify: 833


In [10]:
def result_record_from_row(
    row: pd.Series,
    classification: dict,
    *,
    error: Optional[str] = None,
    finish_reason: str = "",
    raw_response: str = "",
) -> dict:

    rec = {
        "classification_key": clean_scalar(
            row.get("classification_key")
        ),
        "dbcomp_document_id": clean_scalar(
            row.get("dbcomp_document_id")
        ),
        "case_number": clean_scalar(
            row.get("case_number")
        ),
        "raw_file_path": first_nonempty(
            row,
            [
                "raw_file_path",
                "raw_path",
                "local_path",
                "final_local_path",
            ],
        ),
        "clean_text_path": first_nonempty(
            row,
            [
                "clean_text_path",
                "final_text_path",
                "text_path",
            ],
        ),
        "markdown_path": first_nonempty(
            row,
            [
                "markdown_path",
                "final_markdown_path",
            ],
        ),
        "llm_input_path": clean_scalar(
            row.get("llm_input_path")
        ),
        "llm_input_path_source": clean_scalar(
            row.get("llm_input_path_source")
        ),
        "llm_model": MODEL,
        "llm_classified_at": datetime.now(
            timezone.utc
        ).isoformat(timespec="seconds"),
        "llm_finish_reason": finish_reason,
        "llm_error": clean_scalar(error),
        "llm_raw_response": raw_response,
    }

    rec.update(classification)
    return rec


## 7. Parallel Gemma 31B inference

Each row is a separate inference job.

The semaphore limits the number of simultaneous requests while vLLM can batch them internally. Failed requests are retried with exponential backoff.


### Live diagnostics

During classification this version prints only exceptional events:

- `⚠ NO INPUT` — no text file could be resolved
- `⚠ LENGTH` — Gemma hit `MAX_OUTPUT_TOKENS`
- `⚠ RETRY` — HTTP, JSON, schema, or other processing error; job will be retried
- `✖ FAILED` — retries exhausted; row is sent to `REVIEW`

Set `PRINT_REVIEW_ROWS = True` if you also want to print successful responses that end up in `REVIEW` because of confidence/uncertainty.


In [11]:
def _diag_write(message: str) -> None:
    """Write a live diagnostic without badly corrupting the tqdm bar."""
    if not PRINT_LIVE_DIAGNOSTICS:
        return

    try:
        writer = getattr(tqdm, "write", None)
        if callable(writer):
            writer(message)
        else:
            print(message, flush=True)
    except Exception:
        print(message, flush=True)


def _row_label(row: pd.Series) -> str:
    doc_id = clean_scalar(
        row.get("dbcomp_document_id")
    ) or "?"

    case_number = clean_scalar(
        row.get("case_number")
    ) or "?"

    return (
        f"doc={doc_id} | case={case_number}"
    )


async def call_vllm(
    client: httpx.AsyncClient,
    user_prompt: str,
) -> dict[str, str]:

    endpoint = (
        f"{VLLM_BASE_URL.rstrip('/')}"
        "/chat/completions"
    )

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "structured_outputs": {
            "json": RESPONSE_JSON_SCHEMA,
        },
    }

    headers = {
        "Authorization": f"Bearer {VLLM_API_KEY}",
    }

    response = await client.post(
        endpoint,
        json=payload,
        headers=headers,
    )

    response.raise_for_status()
    body = response.json()

    choices = body.get("choices") or []

    if not choices:
        raise ValueError(
            "vLLM response contains no choices"
        )

    choice = choices[0]

    content = (
        choice.get("message", {})
        .get("content")
    )

    if not content:
        raise ValueError(
            "vLLM response contains no message content"
        )

    return {
        "content": str(content),
        "finish_reason": str(
            choice.get("finish_reason")
            or ""
        ),
    }


async def classify_one_row(
    row_dict: dict[str, Any],
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:

    row = pd.Series(row_dict)
    label = _row_label(row)

    if not clean_scalar(
        row.get("llm_input_path")
    ):
        _diag_write(
            f"⚠ NO INPUT | {label} | "
            "No clean text/markdown path resolved -> REVIEW"
        )

        classification = fallback_review(
            "No clean text or markdown file could be resolved for this row."
        )

        return result_record_from_row(
            row,
            classification,
            error="missing_llm_input_path",
        )

    prompt = build_user_prompt(row)
    last_error = ""

    for attempt in range(
        MAX_RETRIES + 1
    ):
        try:
            async with semaphore:
                result = await call_vllm(
                    client,
                    prompt,
                )

            # A technically successful HTTP response can still fail because
            # generation hit MAX_OUTPUT_TOKENS.
            if result["finish_reason"] == "length":
                _diag_write(
                    f"⚠ LENGTH | {label} | "
                    f"attempt={attempt + 1}/{MAX_RETRIES + 1} | "
                    f"hit MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS} -> REVIEW"
                )

                classification = fallback_review(
                    "Model response hit max_tokens; "
                    "marked for review without repeating the same generation."
                )

                return result_record_from_row(
                    row,
                    classification,
                    error="finish_reason=length",
                    finish_reason=result["finish_reason"],
                    raw_response=result["content"],
                )

            # JSON / structured-response errors are caught below and printed.
            parsed = extract_json_object(
                result["content"]
            )

            classification = (
                validate_model_response(
                    parsed
                )
            )

            if (
                PRINT_REVIEW_ROWS
                and classification.get(
                    "pipeline_action"
                ) == "review"
            ):
                _diag_write(
                    f"• REVIEW | {label} | "
                    f"status={classification.get('llm_inclusion_status')} | "
                    f"class={classification.get('llm_document_class')} | "
                    f"confidence={classification.get('llm_confidence')}"
                )

            return result_record_from_row(
                row,
                classification,
                error=None,
                finish_reason=result[
                    "finish_reason"
                ],
                raw_response=result[
                    "content"
                ],
            )

        except Exception as exc:
            last_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if attempt < MAX_RETRIES:
                _diag_write(
                    f"⚠ RETRY | {label} | "
                    f"attempt={attempt + 1}/{MAX_RETRIES + 1} | "
                    f"{last_error}"
                )

                await asyncio.sleep(
                    2 ** attempt
                )

            else:
                _diag_write(
                    f"✖ FAILED | {label} | "
                    f"attempt={attempt + 1}/{MAX_RETRIES + 1} | "
                    f"{last_error} -> REVIEW"
                )

    classification = fallback_review(
        "LLM call, structured output, or JSON parsing failed."
    )

    return result_record_from_row(
        row,
        classification,
        error=last_error,
    )


async def run_parallel_classification(
    todo_df: pd.DataFrame,
) -> Dict[str, dict]:

    latest = load_existing_results(
        JSONL_RESULTS_PATH
    )

    if RUN_MODE == "rebuild_csv":
        return latest

    if todo_df.empty:
        print("No rows to classify.")
        return latest

    timeout = httpx.Timeout(
        REQUEST_TIMEOUT_SECONDS
    )

    limits = httpx.Limits(
        max_connections=max(
            CONCURRENT_REQUESTS * 2,
            8,
        ),
        max_keepalive_connections=max(
            CONCURRENT_REQUESTS,
            4,
        ),
    )

    semaphore = asyncio.Semaphore(
        CONCURRENT_REQUESTS
    )

    row_dicts = todo_df.to_dict(
        orient="records"
    )

    print(
        f"Starting {len(row_dicts):,} jobs | "
        f"concurrency={CONCURRENT_REQUESTS} | "
        f"max_tokens={MAX_OUTPUT_TOKENS} | "
        f"retries={MAX_RETRIES}"
    )

    async with httpx.AsyncClient(
        timeout=timeout,
        limits=limits,
    ) as client:

        tasks = [
            asyncio.create_task(
                classify_one_row(
                    row_dict,
                    client,
                    semaphore,
                )
            )
            for row_dict in row_dicts
        ]

        for completed in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Gemma DB-COMP classification",
            unit="doc",
            mininterval=0.5,
            dynamic_ncols=True,
        ):
            rec = await completed

            append_jsonl_durable(
                JSONL_RESULTS_PATH,
                rec,
            )

            latest[
                rec["classification_key"]
            ] = rec

    return latest


## 8. Optional smoke test

This previews one real prompt. Set `RUN_SMOKE_TEST = True` to send exactly one request without writing it to the JSONL checkpoint.


In [12]:
if len(todo) == 0:
    print("No rows available for a smoke test.")
else:
    sample_row = todo.iloc[0]

    print(
        "classification_key:",
        sample_row["classification_key"],
    )
    print(
        "llm_input_path:",
        sample_row.get("llm_input_path"),
    )

    print("\nPrompt preview:")
    print(
        build_user_prompt(
            sample_row
        )[:5000]
    )

    RUN_SMOKE_TEST = False

    if RUN_SMOKE_TEST:
        timeout = httpx.Timeout(
            REQUEST_TIMEOUT_SECONDS
        )

        async with httpx.AsyncClient(
            timeout=timeout
        ) as client:
            raw = await call_vllm(
                client,
                build_user_prompt(
                    sample_row
                ),
            )

        parsed = extract_json_object(
            raw["content"]
        )
        validated = (
            validate_model_response(
                parsed
            )
        )

        print(
            json.dumps(
                validated,
                ensure_ascii=False,
                indent=2,
            )
        )
    else:
        print(
            "\nSet RUN_SMOKE_TEST=True in this cell "
            "to call Gemma once."
        )


classification_key: 1061||93
llm_input_path: /home/edik/projects/eccjeu/data/processed/com_db_comp/text/1061.txt

Prompt preview:
Metadata:
{"dbcomp_document_id": "1061", "case_number": "93", "case_title": "", "document_title": ""}

Document text:
[page 1] 1 . 79No L 11 / 16 Official Journal of the European Communities 17.

II

(Acts whose publication is not obligatory)

COMMISSION

COMMISSION DECISION

of 7 December 1978

relating to a proceeding under Article 85 of the EEC Treaty (IV/93 — EMO)

(79/37/EEC)

THE COMMISSION OF THE EUROPEAN whereas : COMMUNITIES,

Having regard to the Treaty establishing the European Economic Community, and in particular Article 85 A. The facts thereof,

1 . CECIMO, a de facto association established inHaving regard to Regulation No 17 of 6 February 1950, consists of the following national trade associa1962 (1), and in particular Articles 6 and 8 thereof, tions of machine tool manufacturers :

Having regard to Commission Decision 69/90/EEC (2) Fachverba

## 9. Run classification

In Jupyter, top-level `await` is supported. Each completed record is written immediately to the v2 JSONL checkpoint.


In [13]:
if RUN_MODE == "rebuild_csv":
    print(
        "RUN_MODE=rebuild_csv: "
        "skipping all LLM calls."
    )
    latest_results = (
        load_existing_results(
            JSONL_RESULTS_PATH
        )
    )
else:
    latest_results = (
        await run_parallel_classification(
            todo
        )
    )

print(
    "Checkpoint rows now available:",
    len(latest_results),
)
print("JSONL:", JSONL_RESULTS_PATH)


Starting 833 jobs | concurrency=6 | max_tokens=600 | retries=1


Gemma DB-COMP classification:   0%|          | 0/833 [00:00<?, ?doc/s]

Checkpoint rows now available: 833
JSONL: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_document_classification_v2.jsonl


## 10. Build the v2 DB-COMP metadata manifest

The metadata manifest retains the original DB-COMP columns and adds the v2 classifier fields.

The most useful columns are:

- `pipeline_action`: `keep`, `exclude`, `review`
- `llm_inclusion_status`
- `llm_document_class`
- `llm_decision_type`
- `llm_confidence`
- `llm_reason`
- `llm_evidence_snippet`
- `llm_manual_review`


In [14]:
results = load_existing_results(
    JSONL_RESULTS_PATH
)

class_df = pd.DataFrame(
    results.values()
)

print(
    "Classification rows in JSONL:",
    len(class_df),
)

if class_df.empty:
    print(
        "No classification results available. "
        "Metadata manifest cannot be written yet."
    )

else:
    class_df = (
        class_df
        .sort_values(
            "llm_classified_at"
        )
        .drop_duplicates(
            "classification_key",
            keep="last",
        )
    )

    class_keep = [
        "classification_key",
        "raw_file_path",
        "clean_text_path",
        "markdown_path",
        "llm_input_path",
        "pipeline_action",
        "pipeline_include",
        "pipeline_exclude",
        "llm_inclusion_status",
        "llm_document_class",
        "llm_decision_type",
        "llm_confidence",
        "llm_manual_review",
        "llm_reason",
        "llm_evidence_snippet",
        "llm_error",
        "llm_finish_reason",
        "llm_model",
        "llm_classified_at",
    ]

    class_keep = [
        c
        for c in class_keep
        if c in class_df.columns
    ]

    base = db_comp_manifest.copy()
    base["classification_key"] = (
        base.apply(
            make_row_key,
            axis=1,
        )
    )

    metadata = base.merge(
        class_df[class_keep],
        on="classification_key",
        how="left",
        suffixes=("", "_llm"),
    )

    # Fill source-path fields from the classifier record when the
    # original manifest does not already contain them.
    for col in [
        "raw_file_path",
        "clean_text_path",
        "markdown_path",
    ]:
        alt = f"{col}_llm"

        if alt in metadata.columns:
            if col in metadata.columns:
                metadata[col] = (
                    metadata[col].where(
                        metadata[col]
                        .map(clean_scalar)
                        .astype(bool),
                        metadata[alt],
                    )
                )
                metadata = metadata.drop(
                    columns=[alt]
                )
            else:
                metadata = metadata.rename(
                    columns={alt: col}
                )

    desired_first = [
        "dbcomp_document_id",
        "case_number",
        "case_title",
        "case_name",
        "document_title",
        "title",
        "file_url",
        "download_url",
        "raw_file_path",
        "clean_text_path",
        "markdown_path",

        "pipeline_action",
        "pipeline_include",
        "pipeline_exclude",

        "llm_inclusion_status",
        "llm_document_class",
        "llm_decision_type",
        "llm_confidence",
        "llm_manual_review",
        "llm_reason",
        "llm_evidence_snippet",
        "llm_error",
        "llm_finish_reason",
        "llm_model",
        "llm_classified_at",
    ]

    ordered = []

    for c in desired_first:
        if (
            c in metadata.columns
            and c not in ordered
        ):
            ordered.append(c)

    preserve_original = [
        c
        for c in metadata.columns
        if c not in ordered
        and not c.startswith("llm_")
        and c != "classification_key"
    ]

    metadata = metadata[
        ordered + preserve_original
    ]

    # Stable leading IDs.
    first = [
        c
        for c in [
            "dbcomp_document_id",
            "case_number",
        ]
        if c in metadata.columns
    ]

    rest = [
        c
        for c in metadata.columns
        if c not in first
    ]

    metadata = metadata[
        first + rest
    ]

    metadata.to_csv(
        METADATA_MANIFEST_PATH,
        index=False,
    )

    print(
        "Wrote:",
        METADATA_MANIFEST_PATH,
    )
    print(
        "Rows:",
        len(metadata),
        "| Columns:",
        len(metadata.columns),
    )

    display(
        metadata.head(10)
    )

    summary_cols = [
        c
        for c in [
            "pipeline_action",
            "llm_inclusion_status",
            "llm_document_class",
            "llm_decision_type",
            "llm_manual_review",
        ]
        if c in metadata.columns
    ]

    summary = (
        metadata
        .groupby(
            summary_cols,
            dropna=False,
        )
        .size()
        .reset_index(name="n")
        .sort_values(
            summary_cols
        )
    )

    summary.to_csv(
        SUMMARY_PATH,
        index=False,
    )

    print(
        "Wrote:",
        SUMMARY_PATH,
    )

    display(summary)


Classification rows in JSONL: 833
Wrote: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_metadata_manifest_v2.csv
Rows: 833 | Columns: 36


,dbcomp_document_id,case_number,file_url,raw_file_path,clean_text_path,markdown_path,pipeline_action,pipeline_include,pipeline_exclude,llm_inclusion_status,...,document_name,decision_date,pdf_filename,dbcomp_document_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error
0,1061,93,https://db-comp.eu/document_1061.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,European Machine Tool Exhibitions (EEMO),13 Mar 1969,CELEX-31979D0037-EN-TXT.pdf,https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",93 European Machine Tool Exhibitions (EEMO) CE...,parsed,NaN
1,1064,399904,https://db-comp.eu/document_1064.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Rechargeable battery,12 Dec 2016,399904-rechargeable-battery-_Dec-2016_.pdf,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-_Dec-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",399904 Rechargeable battery 399904-rechargeabl...,parsed,NaN
2,1065,40481,https://db-comp.eu/document_1065.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,40481-Occupant-Safety-Systems.pdf,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40481 Occupant Safety Systems (II) supplied to...,parsed,NaN
3,1066,40360,https://db-comp.eu/document_1066.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Production and distribution of audiobooks,19 Jan 2017,40360-production-and-distribution-of-audiobook...,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40360 Production and distribution of audiobook...,parsed,NaN
4,1067,40291,https://db-comp.eu/document_1067.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Aquatrend,21 Jan 2016,40291-Aquatrend-_Jan-2016_.pdf,https://db-comp.eu/document_1067.download,40291-Aquatrend-_Jan-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40291 Aquatrend 40291-Aquatrend-(Jan-2016).pdf...,parsed,NaN
5,1068,40208,https://db-comp.eu/document_1068.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,International Skating Unionâ€™s Eligibility rules,8 Dec 2017,40208-International-Skating-Union-s-Eligibilit...,https://db-comp.eu/document_1068.download,40208-International-Skating-Union-s-Eligibilit...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40208 International Skating Unionâ€™s Eligibil...,parsed,NaN
6,1069,40169,https://db-comp.eu/document_1069.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,MACO,11 Mar 2016,40169-MACO-_March-2016_.pdf,https://db-comp.eu/document_1069.download,40169-MACO-_March-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40169 MACO 40169-MACO-(March-2016).pdf Agreeme...,parsed,NaN
7,1072,40113,https://db-comp.eu/document_1072.download,/home/edi

Wrote: /home/edik/projects/eccjeu/output/com_db_comp/db_comp_document_classification_summary_v2.csv


,pipeline_action,llm_inclusion_status,llm_document_class,llm_decision_type,llm_manual_review,n
0,exclude,exclude,closure_or_administrative_notice,not_applicable,False,3
1,exclude,exclude,informal_guidance_letter,not_applicable,False,3
2,exclude,exclude,market_test_notice,not_applicable,False,1
3,exclude,exclude,other_non_substantive_document,not_applicable,False,5
4,exclude,exclude,press_release_or_memo,not_applicable,False,59
5,exclude,exclude,rejection_of_complaint,not_applicable,False,107
6,exclude,exclude,summary_or_publication_notice,not_applicable,False,5
7,keep,include,substantive_commission_act,commitment,False,63
8,keep,include,substantive_commission_act,exemption_or_negative_clearance,False,189
9,keep,include,substantive_commission_act,fine_or_periodic_penalty,False,4


## 11. Review views

These views are deliberately action-oriented:

- documents automatically retained
- documents automatically excluded
- documents requiring review
- errors


In [15]:
if "metadata" in globals():

    print("KEEP:")
    display(
        metadata[
            metadata.get(
                "pipeline_action",
                pd.Series(
                    index=metadata.index,
                    dtype=str,
                ),
            ).eq("keep")
        ].head(50)
    )

    print("\nEXCLUDE:")
    display(
        metadata[
            metadata.get(
                "pipeline_action",
                pd.Series(
                    index=metadata.index,
                    dtype=str,
                ),
            ).eq("exclude")
        ].head(50)
    )

    print("\nREVIEW:")
    display(
        metadata[
            metadata.get(
                "pipeline_action",
                pd.Series(
                    index=metadata.index,
                    dtype=str,
                ),
            ).eq("review")
        ].head(50)
    )

    print("\nERRORS:")
    error_series = metadata.get(
        "llm_error",
        pd.Series(
            "",
            index=metadata.index,
            dtype=str,
        ),
    )

    display(
        metadata[
            error_series
            .map(clean_scalar)
            .astype(bool)
        ].head(50)
    )

else:
    print(
        "No metadata manifest "
        "in memory yet."
    )


KEEP:


,dbcomp_document_id,case_number,file_url,raw_file_path,clean_text_path,markdown_path,pipeline_action,pipeline_include,pipeline_exclude,llm_inclusion_status,...,document_name,decision_date,pdf_filename,dbcomp_document_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error
0,1061,93,https://db-comp.eu/document_1061.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,European Machine Tool Exhibitions (EEMO),13 Mar 1969,CELEX-31979D0037-EN-TXT.pdf,https://db-comp.eu/document_1061.download,CELEX-31979D0037-EN-TXT.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",93 European Machine Tool Exhibitions (EEMO) CE...,parsed,NaN
2,1065,40481,https://db-comp.eu/document_1065.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,Occupant Safety Systems (II) supplied to the V...,5 Mar 2019,40481-Occupant-Safety-Systems.pdf,https://db-comp.eu/document_1065.download,40481-Occupant-Safety-Systems.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40481 Occupant Safety Systems (II) supplied to...,parsed,NaN
5,1068,40208,https://db-comp.eu/document_1068.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,International Skating Unionâ€™s Eligibility rules,8 Dec 2017,40208-International-Skating-Union-s-Eligibilit...,https://db-comp.eu/document_1068.download,40208-International-Skating-Union-s-Eligibilit...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40208 International Skating Unionâ€™s Eligibil...,parsed,NaN
7,1072,40113,https://db-comp.eu/document_1072.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,Spark Plugs,21 Feb 2018,40113-Spark-Plugs.pdf,https://db-comp.eu/document_1072.download,40113-Spark-Plugs.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40113 Spark Plugs 40113-Spark-Plugs.pdf Agreem...,parsed,NaN
9,1074,40098,https://db-comp.eu/document_1074.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,Blocktrains,15 Jul 2015,40098-Blocktrains-_July-2015_.pdf,https://db-comp.eu/document_1074.download,40098-Blocktrains-_July-2015_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40098 Blocktrains 40098-Blocktrains-(July-2015...,parsed,NaN
12,1078,40055,https://db-comp.eu/document_1078.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,Parking heaters,17 Jun 2015,40055-Parking-heaters-_June-2015_.pdf,https://db-comp.eu/document_1078.download,40055-Parking-heaters-_June-2015_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40055 Parking heaters 40055-Parking-heaters-(J...,parsed,NaN
14,1080,40049,https://db-comp.eu/document_1080.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,keep,True,False,include,...,Mastercard II,22 Jan 2019,40049-Mastercard-II.pdf,https://db-comp.eu/document_1080.download,40049-Mastercard-II.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40049 Mastercard II 40049-Mastercard-II.pdf Ag...,parsed,NaN
15,1081,40028,https://db-comp.eu/document_1081.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/


EXCLUDE:


,dbcomp_document_id,case_number,file_url,raw_file_path,clean_text_path,markdown_path,pipeline_action,pipeline_include,pipeline_exclude,llm_inclusion_status,...,document_name,decision_date,pdf_filename,dbcomp_document_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error
1,1064,399904,https://db-comp.eu/document_1064.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Rechargeable battery,12 Dec 2016,399904-rechargeable-battery-_Dec-2016_.pdf,https://db-comp.eu/document_1064.download,399904-rechargeable-battery-_Dec-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",399904 Rechargeable battery 399904-rechargeabl...,parsed,NaN
3,1066,40360,https://db-comp.eu/document_1066.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Production and distribution of audiobooks,19 Jan 2017,40360-production-and-distribution-of-audiobook...,https://db-comp.eu/document_1066.download,40360-production-and-distribution-of-audiobook...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40360 Production and distribution of audiobook...,parsed,NaN
4,1067,40291,https://db-comp.eu/document_1067.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Aquatrend,21 Jan 2016,40291-Aquatrend-_Jan-2016_.pdf,https://db-comp.eu/document_1067.download,40291-Aquatrend-_Jan-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40291 Aquatrend 40291-Aquatrend-(Jan-2016).pdf...,parsed,NaN
6,1069,40169,https://db-comp.eu/document_1069.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,MACO,11 Mar 2016,40169-MACO-_March-2016_.pdf,https://db-comp.eu/document_1069.download,40169-MACO-_March-2016_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40169 MACO 40169-MACO-(March-2016).pdf Agreeme...,parsed,NaN
8,1073,40105,https://db-comp.eu/document_1073.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,UEFA Financial Fair Play Rules,24 Oct 2014,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,https://db-comp.eu/document_1073.download,40105-UEFA-Financial-Fair-Play-Rules-_Oct-2014...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40105 UEFA Financial Fair Play Rules 40105-UEF...,parsed,NaN
10,1075,40083,https://db-comp.eu/document_1075.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Irish Distillers,18 Nov 2014,40083-Irish-Distillers-_Nov-2014_.pdf,https://db-comp.eu/document_1075.download,40083-Irish-Distillers-_Nov-2014_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40083 Irish Distillers 40083-Irish-Distillers-...,parsed,NaN
11,1077,40072,https://db-comp.eu/document_1077.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,exclude,False,True,exclude,...,Magyar Suzuki Corporation,14 Oct 2014,40072-Magyar-Suzuki-Corporation-_Oct-2014_.pdf,https://db-comp.eu/document_1077.download,40072-Magyar-Suzuki-Corporation-_Oct-2014_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",40072 Magyar Suzuki Corporation 40072-Magyar-S...,parsed,NaN
13,1079,40050,https://db-comp.eu/document_1079.download,/home/edik/pr


REVIEW:


,dbcomp_document_id,case_number,file_url,raw_file_path,clean_text_path,markdown_path,pipeline_action,pipeline_include,pipeline_exclude,llm_inclusion_status,...,document_name,decision_date,pdf_filename,dbcomp_document_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error
68,1166,39452,https://db-comp.eu/document_1166.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,review,False,False,unclear,...,Mountings for windows and window-doors,28 Mar 2012,39452-Mountings-for-windows-and-window-doors-_...,https://db-comp.eu/document_1166.download,39452-Mountings-for-windows-and-window-doors-_...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",39452 Mountings for windows and window-doors 3...,parsed,NaN
72,1172,39401,https://db-comp.eu/document_1172.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,review,False,False,unclear,...,E.On/GdF,8 Jul 2009,39401-E.ON-GDF-_July-2009_-_english-summary_.pdf,https://db-comp.eu/document_1172.download,39401-E.ON-GDF-_July-2009_-_english-summary_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",39401 E.On/GdF 39401-E.ON-GDF-(July-2009)-(eng...,parsed,NaN
244,1385,35958,https://db-comp.eu/document_1385.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,review,False,False,unclear,...,ATR/BAE,11 Sep 1995,35958-ATR-BAE-_11.9.1995_-_Sep-1995_.pdf,https://db-comp.eu/document_1385.download,35958-ATR-BAE-_11.9.1995_-_Sep-1995_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",35958 ATR/BAE 35958-ATR-BAE-(11.9.1995)-(Sep-1...,parsed,NaN
618,1866,38745,https://db-comp.eu/document_1866.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,review,False,False,unclear,...,BDkEP/Deutsche Post AG,20 Oct 2004,38745-BDkEP-Deutsche-Post-_106_.pdf,https://db-comp.eu/document_1866.download,38745-BDkEP-Deutsche-Post-_106_.pdf,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...",38745 BDkEP/Deutsche Post AG 38745-BDkEP-Deuts...,parsed,NaN
718,1972,39388,https://db-comp.eu/document_1972.download,/home/edik/projects/eccjeu/data/raw/com_db_com...,/home/edik/projects/eccjeu/data/processed/com_...,/home/edik/projects/eccjeu/data/processed/com_...,review,False,False,unclear,...,German Electricity Wholesale Market - German E...,26 Nov 2008,39388-German-Electricity-Wholesale-_Commitment...,https://db-comp.eu/document_1972.download,39388-German-Electricity-Wholesale-_Commitment...,pdf,"[{""text"": """", ""href"": ""https://db-comp.eu/docu...","39389, 39388 German Electricity Wholesale Mark...",parsed,NaN



ERRORS:


,dbcomp_document_id,case_number,file_url,raw_file_path,clean_text_path,markdown_path,pipeline_action,pipeline_include,pipeline_exclude,llm_inclusion_status,...,document_name,decision_date,pdf_filename,dbcomp_document_url,file_name,file_extension,raw_links_json,raw_result_text,scrape_status,scrape_error


## 12. Run information


In [16]:
run_info = {
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(timespec="seconds"),
    "notebook_name": (
        "DB3_classify_db_comp_v5.ipynb"
    ),
    "purpose": (
        "Parallel Gemma 31B classification of DB-COMP documents "
        "into substantive Commission acts versus excluded surrounding material."
    ),
    "project_root": str(PROJECT_ROOT),
    "db_comp_manifest_path": str(
        DB_COMP_MANIFEST_PATH
    ),
    "clean_file_manifest_path": str(
        CLEAN_FILE_MANIFEST_PATH
    ),
    "jsonl_results_path": str(
        JSONL_RESULTS_PATH
    ),
    "metadata_manifest_path": str(
        METADATA_MANIFEST_PATH
    ),
    "summary_path": str(
        SUMMARY_PATH
    ),
    "model": MODEL,
    "vllm_base_url": VLLM_BASE_URL,
    "run_mode": RUN_MODE,
    "test_mode": TEST_MODE,
    "test_limit": TEST_LIMIT,
    "concurrent_requests": CONCURRENT_REQUESTS,
    "max_retries": MAX_RETRIES,
    "max_text_chars": MAX_TEXT_CHARS,
    "head_chars": HEAD_CHARS,
    "tail_chars": TAIL_CHARS,
    "auto_exclude_confidence": AUTO_EXCLUDE_CONFIDENCE,
    "auto_keep_confidence": AUTO_KEEP_CONFIDENCE,
    "inclusion_statuses": INCLUSION_STATUSES,
    "document_classes": DOCUMENT_CLASSES,
    "decision_types": DECISION_TYPES,
}

RUN_INFO_PATH.write_text(
    json.dumps(
        run_info,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        run_info,
        indent=2,
    )
)


{
  "created_at": "2026-08-31T22:43:07+00:00",
  "notebook_name": "DB3_classify_db_comp_v5.ipynb",
  "purpose": "Parallel Gemma 31B classification of DB-COMP documents into substantive Commission acts versus excluded surrounding material.",
  "project_root": "/home/edik/projects/eccjeu",
  "db_comp_manifest_path": "/home/edik/projects/eccjeu/output/com_db_comp/db_comp_manifest.csv",
  "clean_file_manifest_path": "/home/edik/projects/eccjeu/output/com_db_comp/db_comp_clean_file_manifest.csv",
  "jsonl_results_path": "/home/edik/projects/eccjeu/output/com_db_comp/db_comp_document_classification_v2.jsonl",
  "metadata_manifest_path": "/home/edik/projects/eccjeu/output/com_db_comp/db_comp_metadata_manifest_v2.csv",
  "summary_path": "/home/edik/projects/eccjeu/output/com_db_comp/db_comp_document_classification_summary_v2.csv",
  "model": "google/gemma-4-31B-it",
  "vllm_base_url": "http://localhost:8000/v1",
  "run_mode": "update",
  "test_mode": false,
  "test_limit": 10,
  "concurrent_re